In [1]:
import pandas as pd
from sqlalchemy import create_engine, text

DATABASE_URL = "postgresql+psycopg:///orlando_demand_revenue"

engine = create_engine(DATABASE_URL)

query = text("""
    SELECT *
    FROM analytics.vw_forecast_input
    ORDER BY calendar_date
""")

with engine.connect() as connection:
    forecast_input = pd.read_sql_query(
        query,
        connection,
        parse_dates=["calendar_date"],
    )

forecast_input.shape

(1126, 27)

In [2]:
forecast_input.head()

,date_key,calendar_date,target_demand,has_actuals,day_of_week,day_of_year,week_of_year,month_number,quarter_number,year_number,...,planned_price_multiplier,active_campaign_count,maximum_planned_discount,paid_media_flag,email_campaign_flag,bundle_campaign_flag,discount_campaign_flag,available_capacity,demand_target,revenue_target
0,20230101,2023-01-01,1984.0,True,7,1,52,1,1,2023,...,1.18,0,0.0,0,0,0,0,2500,2379,298665.75
1,20230102,2023-01-02,1369.0,True,1,2,1,1,1,2023,...,1.12,0,0.0,0,0,0,0,2500,1501,178857.72
2,20230103,2023-01-03,849.0,True,2,3,1,1,1,2023,...,0.95,0,0.0,0,0,0,0,2350,853,86214.76
3,20230104,2023-01-04,850.0,True,3,4,1,1,1,2023,...,0.95,0,0.0,0,0,0,0,2350,875,88438.35
4,20230105,2023-01-05,855.0,True,4,5,1,1,1,2023,...,0.95,0,0.0,0,0,0,0,2350,963,97332.72


In [14]:
forecast_input.tail()

,date_key,calendar_date,target_demand,has_actuals,day_of_week,day_of_year,week_of_year,month_number,quarter_number,year_number,...,planned_price_multiplier,active_campaign_count,maximum_planned_discount,paid_media_flag,email_campaign_flag,bundle_campaign_flag,discount_campaign_flag,available_capacity,demand_target,revenue_target
1121,20260126,2026-01-26,NaN,False,1,26,5,1,1,2026,...,0.95,0,0.0,0,0,0,0,2350,962,97231.65
1122,20260127,2026-01-27,NaN,False,2,27,5,1,1,2026,...,0.95,0,0.0,0,0,0,0,2350,915,92481.25
1123,20260128,2026-01-28,NaN,False,3,28,5,1,1,2026,...,0.95,0,0.0,0,0,0,0,2350,938,94805.91
1124,20260129,2026-01-29,NaN,False,4,29,5,1,1,2026,...,0.95,0,0.0,0,0,0,0,2350,1032,104306.72
1125,20260130,2026-01-30,NaN,False,5,30,5,1,1,2026,...,0.95,0,0.0,0,0,0,0,2350,1220,123308.33


In [15]:
forecast_input.columns.tolist()

['date_key',
 'calendar_date',
 'target_demand',
 'has_actuals',
 'day_of_week',
 'day_of_year',
 'week_of_year',
 'month_number',
 'quarter_number',
 'year_number',
 'is_weekend',
 'holiday_flag',
 'school_break_flag',
 'season',
 'observed_temperature_f',
 'observed_precipitation_in',
 'observed_severe_weather_flag',
 'planned_price_multiplier',
 'active_campaign_count',
 'maximum_planned_discount',
 'paid_media_flag',
 'email_campaign_flag',
 'bundle_campaign_flag',
 'discount_campaign_flag',
 'available_capacity',
 'demand_target',
 'revenue_target']

In [16]:
historical = forecast_input[
    forecast_input["target_demand"].notna()
].copy()

future = forecast_input[
    forecast_input["target_demand"].isna()
].copy()

print(
    "Historical:",
    len(historical),
    historical["calendar_date"].min(),
    "to",
    historical["calendar_date"].max(),
)

print(
    "Future:",
    len(future),
    future["calendar_date"].min(),
    "to",
    future["calendar_date"].max(),
)

Historical: 1096 2023-01-01 00:00:00 to 2025-12-31 00:00:00
Future: 30 2026-01-01 00:00:00 to 2026-01-30 00:00:00


In [17]:
assert forecast_input["calendar_date"].is_unique
assert forecast_input["calendar_date"].is_monotonic_increasing
assert forecast_input["calendar_date"].diff().dropna().eq(
    pd.Timedelta(days=1)
).all()

assert len(historical) == 1096
assert len(future) == 30
assert historical["calendar_date"].max() < future["calendar_date"].min()

assert future["observed_temperature_f"].isna().all()
assert future["observed_precipitation_in"].isna().all()

print("Forecast input validation passed.")

Forecast input validation passed.
